<a href="https://colab.research.google.com/github/FRJackson/MBS-first_vs1/blob/main/API_CITY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
%%bash
set -e

PROJECT_DIR="cd-rate-api"
rm -rf "$PROJECT_DIR"
mkdir -p "$PROJECT_DIR/services"

cat > "$PROJECT_DIR/requirements.txt" <<'EOF'
Flask>=2.2
EOF

cat > "$PROJECT_DIR/rates.py" <<'EOF'
"""rates.py

Static rate table for Large Denomination Negotiable CD (prototype).

IMPORTANT
- The rates in this file are SAMPLE DATA for prototype/demo only.
- They are intentionally set in a ~2.0% - 3.0% range for demo realism.
- This is NOT real Citi pricing.

Design
- Support flexible term inputs (any positive integer months).
- System maps the input term into a TERM_BUCKET, then maps amount into an AMOUNT_TIER.
"""

# Term buckets are inclusive on both ends.
# max_months=None means no upper bound.
RATE_TABLE = [
    {
        "min_months": 1,
        "max_months": 2,
        "tiers": [
            {"min": 0,         "max": 249_999,  "rate": 0.0200},
            {"min": 250_000,   "max": 999_999,  "rate": 0.0205},
            {"min": 1_000_000, "max": None,     "rate": 0.0210},
        ],
    },
    {
        "min_months": 3,
        "max_months": 5,
        "tiers": [
            {"min": 0,         "max": 249_999,  "rate": 0.0210},
            {"min": 250_000,   "max": 999_999,  "rate": 0.0215},
            {"min": 1_000_000, "max": None,     "rate": 0.0220},
        ],
    },
    {
        "min_months": 6,
        "max_months": 11,
        "tiers": [
            {"min": 0,         "max": 249_999,  "rate": 0.0220},
            {"min": 250_000,   "max": 999_999,  "rate": 0.0225},
            {"min": 1_000_000, "max": None,     "rate": 0.0230},
        ],
    },
    {
        "min_months": 12,
        "max_months": 17,
        "tiers": [
            {"min": 0,         "max": 249_999,  "rate": 0.0235},
            {"min": 250_000,   "max": 999_999,  "rate": 0.0245},
            {"min": 1_000_000, "max": None,     "rate": 0.0255},
        ],
    },
    {
        "min_months": 18,
        "max_months": 23,
        "tiers": [
            {"min": 0,         "max": 249_999,  "rate": 0.0250},
            {"min": 250_000,   "max": 999_999,  "rate": 0.0260},
            {"min": 1_000_000, "max": None,     "rate": 0.0270},
        ],
    },
    {
        "min_months": 24,
        "max_months": 35,
        "tiers": [
            {"min": 0,         "max": 249_999,  "rate": 0.0260},
            {"min": 250_000,   "max": 999_999,  "rate": 0.0270},
            {"min": 1_000_000, "max": None,     "rate": 0.0280},
        ],
    },
    {
        "min_months": 36,
        "max_months": 59,
        "tiers": [
            {"min": 0,         "max": 249_999,  "rate": 0.0270},
            {"min": 250_000,   "max": 999_999,  "rate": 0.0280},
            {"min": 1_000_000, "max": None,     "rate": 0.0290},
        ],
    },
    {
        "min_months": 60,
        "max_months": None,
        "tiers": [
            {"min": 0,         "max": 249_999,  "rate": 0.0275},
            {"min": 250_000,   "max": 999_999,  "rate": 0.0285},
            {"min": 1_000_000, "max": None,     "rate": 0.0295},
        ],
    },
]
EOF

cat > "$PROJECT_DIR/services/rate_service.py" <<'EOF'
from __future__ import annotations

from dataclasses import dataclass
from decimal import Decimal, InvalidOperation, ROUND_HALF_UP, getcontext
from typing import Optional, Tuple, Dict, Any, List

from rates import RATE_TABLE

# High precision for intermediate calculations
getcontext().prec = 28


@dataclass(frozen=True)
class BucketMatch:
    bucket_min_months: int
    bucket_max_months: Optional[int]


@dataclass(frozen=True)
class TierMatch:
    tier_min: Decimal
    tier_max: Optional[Decimal]
    apr: Decimal


def parse_term_months(value) -> int:
    """Parse term_months from JSON input."""
    try:
        term = int(value)
    except Exception:
        raise ValueError("term_months 必须是整数，例如 6 / 12 / 18。")

    if term <= 0:
        raise ValueError("term_months 必须大于 0。")

    # Keep it flexible; still guard against absurd values.
    if term > 600:
        raise ValueError("term_months 过大（>600），请检查输入。")

    return term


def parse_amount(value) -> Decimal:
    """Parse amount from JSON input. Accepts number or string."""
    try:
        amt = Decimal(str(value))
    except (InvalidOperation, TypeError):
        raise ValueError("amount 必须是数字或可转为数字的字符串，例如 250000 或 '250000'。")

    if amt <= 0:
        raise ValueError("amount 必须大于 0。")

    return amt.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)


def list_term_buckets() -> List[Dict[str, Any]]:
    """Return term bucket metadata (for docs/demo)."""
    buckets = []
    for b in RATE_TABLE:
        buckets.append({
            "min_months": b["min_months"],
            "max_months": b["max_months"],
        })
    return buckets


def bucket_label(min_m: int, max_m: Optional[int]) -> str:
    if max_m is None:
        return f"{min_m}个月及以上"
    if min_m == max_m:
        return f"{min_m}个月"
    return f"{min_m}-{max_m}个月"


def find_bucket(term_months: int) -> BucketMatch:
    """Map term_months into a bucket."""
    for b in RATE_TABLE:
        mn = int(b["min_months"])
        mx = b["max_months"]
        if mx is None:
            if term_months >= mn:
                return BucketMatch(bucket_min_months=mn, bucket_max_months=None)
        else:
            mx_i = int(mx)
            if mn <= term_months <= mx_i:
                return BucketMatch(bucket_min_months=mn, bucket_max_months=mx_i)

    # Should not happen if the last bucket is open-ended.
    raise LookupError("No matching term bucket found.")


def find_tier_for_bucket(bucket: BucketMatch, amount: Decimal) -> TierMatch:
    """Within a term bucket, map amount into a tier and return APR."""
    # Find the bucket entry
    bucket_entry = None
    for b in RATE_TABLE:
        if b["min_months"] == bucket.bucket_min_months and b["max_months"] == bucket.bucket_max_months:
            bucket_entry = b
            break

    if bucket_entry is None:
        raise LookupError("Bucket definition missing.")

    for t in bucket_entry["tiers"]:
        tmin = Decimal(str(t["min"]))
        tmax = None if t["max"] is None else Decimal(str(t["max"]))
        if amount >= tmin and (tmax is None or amount <= tmax):
            apr = Decimal(str(t["rate"]))
            return TierMatch(tier_min=tmin, tier_max=tmax, apr=apr)

    raise LookupError("No matching amount tier found.")


def calc_simple_interest(amount: Decimal, apr: Decimal, term_months: int) -> Tuple[Decimal, Decimal, Dict[str, Any]]:
    """Simple interest per requirement:

    interest = amount * apr * (term_months / 12)

    Additionally, return a `year_fraction` object that reflects an (estimated) Actual/365 day logic.
    We do NOT ask for deposit/withdraw dates in this prototype.

    To align with the (term_months/12) formula while still expressing "days", we estimate:
      estimated_days = term_months * (365/12)
    so that:
      estimated_days / 365 == term_months / 12
    """
    year_fraction = Decimal(term_months) / Decimal(12)

    # Estimated interest days under an "Actual/365"-style representation
    estimated_days = (Decimal(term_months) * Decimal(365) / Decimal(12))

    interest = (amount * apr * year_fraction).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
    maturity_amount = (amount + interest).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)

    year_fraction_obj: Dict[str, Any] = {
        "value": float(year_fraction),
        "fraction_text": f"{term_months}/12",
        "day_count_convention": "Actual/365 (estimated from months; using 365/12 days per month)",
        "estimated_interest_days": float(estimated_days.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)),
        "days_over_365_text": f"{estimated_days.quantize(Decimal('0.01'), rounding=ROUND_HALF_UP)}/365",
    }

    return interest, maturity_amount, year_fraction_obj


def quote_cd(term_months: int, amount: Decimal, as_of: str) -> Dict[str, Any]:
    bucket = find_bucket(term_months)
    tier = find_tier_for_bucket(bucket, amount)

    interest, maturity_amount, year_fraction_obj = calc_simple_interest(amount, tier.apr, term_months)

    return {
        "product": "Large Denomination Negotiable CD (Prototype)",
        "as_of": as_of,
        "term_months": term_months,
        "amount": str(amount),
        "rate": float(tier.apr),
        "rate_type": "APR",
        "matched_term_bucket": {
            "min_months": bucket.bucket_min_months,
            "max_months": bucket.bucket_max_months,
            "label": bucket_label(bucket.bucket_min_months, bucket.bucket_max_months),
        },
        "matched_amount_tier": {
            "tier_min": str(tier.tier_min),
            "tier_max": None if tier.tier_max is None else str(tier.tier_max),
        },
        "interest": str(interest),
        "maturity_amount": str(maturity_amount),
        "year_fraction": year_fraction_obj,
        "calculation": {
            "model": "simple_interest",
            "formula": "interest = principal * APR * (term_months / 12)",
            "rounding": "0.01 (ROUND_HALF_UP)",
        },
        "disclaimer": "Prototype only. Rates are sample data, not real Citi rates.",
    }
EOF

cat > "$PROJECT_DIR/app.py" <<'EOF'
from __future__ import annotations

import os
from datetime import datetime

from flask import Flask, jsonify, request

try:
    from zoneinfo import ZoneInfo
except Exception:  # pragma: no cover
    ZoneInfo = None

from services.rate_service import (
    parse_amount,
    parse_term_months,
    quote_cd,
    list_term_buckets,
    bucket_label,
)


def now_date_iso(tz_name: str = "America/New_York") -> str:
    # For prototype, return date string (YYYY-MM-DD) in a configured timezone.
    try:
        if ZoneInfo is None:
            raise RuntimeError("zoneinfo not available")
        return datetime.now(ZoneInfo(tz_name)).date().isoformat()
    except Exception:
        return datetime.utcnow().date().isoformat()


def make_error(status: int, code: str, message: str):
    return jsonify({"error": {"code": code, "message": message}}), status


app = Flask(__name__)


@app.get("/health")
def health():
    return jsonify({"status": "ok"})


@app.get("/")
@app.get("/demo")
def demo():
    buckets = list_term_buckets()
    bucket_text = ", ".join([bucket_label(b["min_months"], b["max_months"]) for b in buckets])

    return (
        f"""
<!doctype html>
<html lang="zh-CN">
<head>
  <meta charset="utf-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1" />
  <title>CD 利率查询 Demo</title>
  <style>
    body {{ font-family: Arial, sans-serif; margin: 24px; line-height: 1.5; }}
    .row {{ margin: 12px 0; }}
    label {{ display: inline-block; width: 90px; }}
    input {{ padding: 6px 8px; width: 240px; }}
    button {{ padding: 8px 14px; cursor: pointer; }}
    pre {{ background: #f6f8fa; padding: 12px; border-radius: 8px; overflow: auto; }}
    .hint {{ color: #666; font-size: 12px; }}
    code {{ background: #f0f0f0; padding: 2px 6px; border-radius: 6px; }}
  </style>
</head>
<body>
  <h2>大额可转让定期存单（CD）利率查询 - Demo</h2>
  <p class="hint">这是一个原型页面：输入 <code>term_months</code> 与 <code>amount</code> 后点击查询，调用后端 API 并展示 JSON。</p>

  <div class="row">
    <label>期限(月)</label>
    <input id="term" value="18" />
    <span class="hint">支持任意正整数（月）。系统会自动匹配期限区间：{bucket_text}</span>
  </div>

  <div class="row">
    <label>金额</label>
    <input id="amount" value="250000" />
  </div>

  <div class="row">
    <button onclick="quote()">查询</button>
  </div>

  <h3>返回 JSON</h3>
  <pre id="out">{{}}</pre>

<script>
  async function quote() {{
    const term = parseInt(document.getElementById('term').value, 10);
    const amount = document.getElementById('amount').value;

    const resp = await fetch('/api/v1/cd-rate/quote', {{
      method: 'POST',
      headers: {{'Content-Type': 'application/json'}},
      body: JSON.stringify({{term_months: term, amount: amount}})
    }});

    const data = await resp.json();
    document.getElementById('out').textContent = JSON.stringify(data, null, 2);
  }}
</script>
</body>
</html>
        """
    )


@app.get("/api/v1/cd-rate/metadata")
def metadata():
    """Optional helper endpoint for internal demo/docs."""
    return jsonify({
        "product": "Large Denomination Negotiable CD (Prototype)",
        "rate_type": "APR",
        "term_buckets": list_term_buckets(),
        "note": "term_months 支持任意正整数（月），系统会自动匹配到对应期限区间并再按金额档位匹配利率。",
        "disclaimer": "Prototype only. Rates are sample data, not real Citi rates.",
    })


@app.post("/api/v1/cd-rate/quote")
def cd_quote():
    if not request.is_json:
        return make_error(400, "INVALID_REQUEST", "Content-Type 必须为 application/json。")

    payload = request.get_json(silent=True) or {}

    try:
        term_months = parse_term_months(payload.get("term_months", None))
    except ValueError as e:
        return make_error(400, "INVALID_TERM", str(e))

    try:
        amount = parse_amount(payload.get("amount", None))
    except ValueError as e:
        return make_error(400, "INVALID_AMOUNT", str(e))

    tz_name = os.getenv("APP_TZ", "America/New_York")
    as_of = now_date_iso(tz_name)

    try:
        result = quote_cd(term_months=term_months, amount=amount, as_of=as_of)
    except LookupError as e:
        return make_error(404, "NO_RATE", str(e))

    return jsonify(result)


if __name__ == "__main__":
    port = int(os.getenv("PORT", "5000"))
    app.run(host="0.0.0.0", port=port, debug=False, use_reloader=False)
EOF

cat > "$PROJECT_DIR/README.md" <<'EOF'
# 大额可转让定期存单（CD）利率查询 API 原型

> **用途**：内部展示用后端 API 原型（Python + Flask）。
>
> **重要声明**：本项目中的利率为 **示例静态数据**，仅用于原型/演示，不代表任何真实银行报价。

---

## 1. 功能概览

- 输入：`term_months`（存款期限/月）与 `amount`（本金）
- 输出：
  - 自动匹配到的期限区间（Term Bucket）
  - 自动匹配到的金额档位（Amount Tier）
  - 年利率（APR）
  - 预计利息、到期本息
- 利率数据：应用内部静态表（见 `rates.py`）

本原型特意支持 **灵活期限输入**：`term_months` 可为任意正整数（如 18 个月），系统会自动落到期限区间并匹配利率。

---

## 2. 快速启动

### 2.1 安装依赖

```bash
pip install -r requirements.txt

bash: line 469: warning: here-document at line 441 delimited by end-of-file (wanted `EOF')


In [5]:
!pip -q install -r cd-rate-api/requirements.txt

In [6]:
!python cd-rate-api/app.py > cd-rate-api/flask.log 2>&1 &
!sleep 1
!tail -n 40 cd-rate-api/flask.log

 * Serving Flask app 'app'
 * Debug mode: off
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
Press CTRL+C to quit


In [7]:
import requests, json

base = "http://127.0.0.1:5000"

print("Health:", requests.get(f"{base}/health").status_code, requests.get(f"{base}/health").json())

# 18个月示例
payload = {"term_months": 18, "amount": 250000}
r = requests.post(f"{base}/api/v1/cd-rate/quote", json=payload)
print("Quote status:", r.status_code)
print(json.dumps(r.json(), indent=2, ensure_ascii=False))

# 任意期限也可以：比如 19个月
payload2 = {"term_months": 19, "amount": 250000}
r2 = requests.post(f"{base}/api/v1/cd-rate/quote", json=payload2)
print("19m status:", r2.status_code)
print(json.dumps(r2.json(), indent=2, ensure_ascii=False))

# 元数据端点（可选）
m = requests.get(f"{base}/api/v1/cd-rate/metadata")
print("Metadata:", m.status_code)
print(json.dumps(m.json(), indent=2, ensure_ascii=False))

Health: 200 {'status': 'ok'}
Quote status: 200
{
  "amount": "250000.00",
  "as_of": "2026-03-03",
  "calculation": {
    "formula": "interest = principal * APR * (term_months / 12)",
    "model": "simple_interest",
    "rounding": "0.01 (ROUND_HALF_UP)"
  },
  "disclaimer": "Prototype only. Rates are sample data, not real Citi rates.",
  "interest": "9750.00",
  "matched_amount_tier": {
    "tier_max": "999999",
    "tier_min": "250000"
  },
  "matched_term_bucket": {
    "label": "18-23个月",
    "max_months": 23,
    "min_months": 18
  },
  "maturity_amount": "259750.00",
  "product": "Large Denomination Negotiable CD (Prototype)",
  "rate": 0.026,
  "rate_type": "APR",
  "term_months": 18,
  "year_fraction": {
    "day_count_convention": "Actual/365 (estimated from months; using 365/12 days per month)",
    "days_over_365_text": "547.50/365",
    "estimated_interest_days": 547.5,
    "fraction_text": "18/12",
    "value": 1.5
  }
}
19m status: 200
{
  "amount": "250000.00",
  "as_of"

In [8]:
from google.colab import output
output.serve_kernel_port_as_window(5000)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [10]:
!rm -f cd-rate-api.zip
!zip -r cd-rate-api.zip cd-rate-api -x "*__pycache__*" "*.pyc" "*.log"
!ls -lh cd-rate-api.zip

  adding: cd-rate-api/ (stored 0%)
  adding: cd-rate-api/app.py (deflated 52%)
  adding: cd-rate-api/requirements.txt (stored 0%)
  adding: cd-rate-api/README.md (deflated 29%)
  adding: cd-rate-api/rates.py (deflated 79%)
  adding: cd-rate-api/services/ (stored 0%)
  adding: cd-rate-api/services/rate_service.py (deflated 65%)
-rw-r--r-- 1 root root 6.6K Mar  4 01:58 cd-rate-api.zip


In [11]:
from google.colab import files
files.download("cd-rate-api.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>